# Alignement Procruste — Espace Concurrentiel vs Espace Sémantique

Ce notebook aligne les deux projections 2D (concurrence et sémantique) via une transformation de Procruste.

Objectif : mesurer, pour chaque entreprise présente dans les deux analyses, la distance résiduelle entre ses positions après alignement.

## 1. Configuration

Dépendances nécessaires :
```bash
pip install pandas numpy scipy plotly
```

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.spatial import procrustes

ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
EXPORTS_DIR = ROOT / "analyses" / "exports"

COMPETITION_COORDS_PATH = EXPORTS_DIR / "coords_2d.csv"
SEMANTIC_COORDS_PATH = EXPORTS_DIR / "semantic_coords_2d.csv"

print(f"Competition coords: {COMPETITION_COORDS_PATH}")
print(f"Semantic coords:    {SEMANTIC_COORDS_PATH}")

Competition coords: C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv
Semantic coords:    C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\semantic_coords_2d.csv


## 2. Chargement des deux espaces et intersection des entreprises

Restriction appliquée : uniquement les entreprises présentes dans les deux analyses.

In [14]:
from pathlib import Path

# Fallbacks possibles si le nom du fichier a changé
semantic_candidates = [
    SEMANTIC_COORDS_PATH,
    EXPORTS_DIR / "semantic_coordinates_2d.csv",
    EXPORTS_DIR / "semantic_coords.csv",
]

semantic_existing = next((p for p in semantic_candidates if Path(p).exists()), None)

if not COMPETITION_COORDS_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable: {COMPETITION_COORDS_PATH}\n"
        "Exécute d'abord le notebook d'analyse concurrentielle pour générer coords_2d.csv."
    )

if semantic_existing is None:
    expected = "\n - " + "\n - ".join(str(p) for p in semantic_candidates)
    raise FileNotFoundError(
        "Aucun fichier de coordonnées sémantiques trouvé.\n"
        f"Chemins testés:{expected}\n\n"
        "Action requise:\n"
        "1) Ouvre analyses/semantic_similarity_analysis.ipynb\n"
        "2) Exécute les cellules jusqu'à la section MDS (génération de semantic_coords_2d.csv)\n"
        "3) Reviens ici et relance cette cellule."
    )

df_comp = pd.read_csv(COMPETITION_COORDS_PATH)
df_sem = pd.read_csv(semantic_existing)
print(f"Fichier sémantique utilisé: {semantic_existing.name}")

# Harmonisation du nom de colonne entreprise
if "actor" in df_comp.columns:
    df_comp = df_comp.rename(columns={"actor": "name"})

required_cols_comp = {"name", "x", "y"}
required_cols_sem = {"name", "x", "y"}

missing_comp = required_cols_comp - set(df_comp.columns)
missing_sem = required_cols_sem - set(df_sem.columns)

if missing_comp:
    raise ValueError(f"Colonnes manquantes dans coords_2d.csv: {missing_comp}")
if missing_sem:
    raise ValueError(f"Colonnes manquantes dans {semantic_existing.name}: {missing_sem}")

df_comp = df_comp[["name", "x", "y"]].copy()
df_sem = df_sem[["name", "x", "y"]].copy()

common_names = sorted(set(df_comp["name"]).intersection(set(df_sem["name"])))

if len(common_names) == 0:
    raise ValueError(
        "Aucune entreprise commune entre les deux espaces. "
        "Vérifie que les mêmes règles de nommage/normalisation ont été appliquées."
    )

df_comp_common = df_comp[df_comp["name"].isin(common_names)].copy()
df_sem_common = df_sem[df_sem["name"].isin(common_names)].copy()

# Ordre strictement identique pour l'alignement
df_comp_common = df_comp_common.set_index("name").loc[common_names].reset_index()
df_sem_common = df_sem_common.set_index("name").loc[common_names].reset_index()

print(f"Entreprises dans l'espace concurrentiel : {len(df_comp)}")
print(f"Entreprises dans l'espace sémantique   : {len(df_sem)}")
print(f"Entreprises communes retenues          : {len(common_names)}")

df_comp_common.head()

Fichier sémantique utilisé: semantic_coords_2d.csv
Entreprises dans l'espace concurrentiel : 293
Entreprises dans l'espace sémantique   : 84
Entreprises communes retenues          : 40


,name,x,y
0,AMD,-0.392984,-0.280785
1,ARM Holdings,-0.070182,-0.117021
2,Adobe,-0.512149,-0.068497
3,Aleph Alpha,-0.096260,-0.411202
4,Alibaba,-0.063116,0.441501


## 3. Alignement de Procruste

La méthode de Procruste standardise, translate, met à l'échelle et fait pivoter les nuages de points pour minimiser leur écart global.

In [15]:
X_comp = df_comp_common[["x", "y"]].to_numpy()
Y_sem = df_sem_common[["x", "y"]].to_numpy()

X_aligned, Y_aligned, disparity = procrustes(X_comp, Y_sem)

# Distance residuelle par entreprise apres alignement
residual_dist = np.linalg.norm(X_aligned - Y_aligned, axis=1)

df_procrustes = pd.DataFrame({
    "name": common_names,
    "x_comp_aligned": X_aligned[:, 0],
    "y_comp_aligned": X_aligned[:, 1],
    "x_sem_aligned": Y_aligned[:, 0],
    "y_sem_aligned": Y_aligned[:, 1],
    "residual_distance": residual_dist
}).sort_values("residual_distance", ascending=False).reset_index(drop=True)

print(f"Disparity globale Procruste : {disparity:.6f}")
print("\nStatistiques des distances residuelles :")
display(df_procrustes["residual_distance"].describe())

print("\nTop 20 entreprises avec le plus grand ecart inter-espaces :")
display(df_procrustes.head(20))

Disparity globale Procruste : 0.960928

Statistiques des distances residuelles :


count    40.000000
mean      0.137194
std       0.073036
min       0.022940
25%       0.086452
50%       0.127918
75%       0.204798
max       0.278731
Name: residual_distance, dtype: float64


Top 20 entreprises avec le plus grand ecart inter-espaces :


,name,x_comp_aligned,y_comp_aligned,x_sem_aligned,y_sem_aligned,residual_distance
0,ManoMano,0.217966,-0.150426,-0.000523,0.022644,0.278731
1,Aqemia,0.253325,-0.024954,-0.017996,-0.025377,0.271322
2,BenevolentAI,0.142898,-0.200162,-0.025024,-0.006463,0.256354
3,Prophesee,-0.079363,0.201039,-0.020990,-0.019974,0.228591
4,Ant Group,-0.165697,0.126977,-0.006922,-0.035643,0.227276
5,Databricks,0.157391,0.184960,0.028743,0.014476,0.213577
6,Comand AI,-0.071477,0.211172,0.010606,0.015611,0.212089
7,Y combinator,0.019251,-0.240712,0.018339,-0.030315,0.210399
8,Scale AI,-0.202636,0.024119,0.006148,0.010839,0.209206
9,Appen,-0.113687,0.168630,0.016981,0.009992,0.205524


## 4. Visualisation des écarts après alignement

Chaque segment relie la position d'une entreprise dans l'espace concurrentiel aligné et dans l'espace sémantique aligné.

In [16]:
fig = go.Figure()

# Segments entreprise par entreprise
for _, r in df_procrustes.iterrows():
    fig.add_trace(go.Scatter(
        x=[r["x_comp_aligned"], r["x_sem_aligned"]],
        y=[r["y_comp_aligned"], r["y_sem_aligned"]],
        mode="lines",
        line=dict(color="rgba(120,120,120,0.25)", width=1),
        hoverinfo="skip",
        showlegend=False
    ))

# Points espace concurrentiel aligné
fig.add_trace(go.Scatter(
    x=df_procrustes["x_comp_aligned"],
    y=df_procrustes["y_comp_aligned"],
    mode="markers",
    marker=dict(size=7, color="#2E8B57", opacity=0.85),
    name="Espace concurrentiel (aligné)",
    text=df_procrustes["name"],
    hovertemplate="<b>%{text}</b><br>Comp aligné<extra></extra>"
))

# Points espace sémantique aligné
fig.add_trace(go.Scatter(
    x=df_procrustes["x_sem_aligned"],
    y=df_procrustes["y_sem_aligned"],
    mode="markers",
    marker=dict(size=7, color="#D2691E", opacity=0.85),
    name="Espace sémantique (aligné)",
    text=df_procrustes["name"],
    hovertemplate="<b>%{text}</b><br>Sémantique aligné<br>Distance: %{customdata:.4f}<extra></extra>",
    customdata=df_procrustes["residual_distance"]
))

fig.update_layout(
    title=f"Alignement de Procruste — {len(df_procrustes)} entreprises communes",
    xaxis_title="Dimension 1 (espace aligné)",
    yaxis_title="Dimension 2 (espace aligné)",
    width=1400,
    height=1000,
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    font=dict(family="Inter, sans-serif", size=11),
    hovermode="closest"
)

fig.show()

## 5. Exports

Exports produits :
- positions alignées et distance résiduelle par entreprise
- top entreprises à plus fort écart
- résumé statistique

In [17]:
aligned_path = EXPORTS_DIR / "procrustes_aligned_positions.csv"
top_gap_path = EXPORTS_DIR / "procrustes_top_gaps.csv"
summary_path = EXPORTS_DIR / "procrustes_summary.csv"

df_procrustes.to_csv(aligned_path, index=False)
df_procrustes.head(50).to_csv(top_gap_path, index=False)

summary = pd.DataFrame([{
    "n_common_companies": len(df_procrustes),
    "procrustes_disparity": disparity,
    "mean_residual_distance": df_procrustes['residual_distance'].mean(),
    "median_residual_distance": df_procrustes['residual_distance'].median(),
    "p90_residual_distance": df_procrustes['residual_distance'].quantile(0.90),
    "max_residual_distance": df_procrustes['residual_distance'].max()
}])
summary.to_csv(summary_path, index=False)

print("Exports générés :")
for p in [aligned_path, top_gap_path, summary_path]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<36} {size_kb:7.1f} KB")

display(summary)

Exports générés :
  procrustes_aligned_positions.csv         4.5 KB
  procrustes_top_gaps.csv                  4.5 KB
  procrustes_summary.csv                   0.2 KB


,n_common_companies,procrustes_disparity,mean_residual_distance,median_residual_distance,p90_residual_distance,max_residual_distance
0,40,0.960928,0.137194,0.127918,0.227408,0.278731


## 6. Interprétation rapide

- **Distance résiduelle faible** : cohérence forte entre voisinage concurrentiel et voisinage sémantique.
- **Distance résiduelle élevée** : décalage entre positionnement concurrentiel observé et proximité des descriptions (possible spécialisation, positionnement narratif, ou données à enrichir).
- Les entreprises du **top gaps** sont prioritaires pour revue qualitative.

## 7. Variante robuste (normalisation quantile) et comparaison

Cette variante applique une normalisation robuste (médiane + IQR) sur les deux espaces avant l'alignement de Procruste.

But : tester la sensibilité des distances résiduelles aux valeurs extrêmes de projection.

In [18]:
def robust_scale_iqr(arr: np.ndarray) -> np.ndarray:
    """Scale 2D coordinates with median/IQR per axis."""
    med = np.median(arr, axis=0)
    q1 = np.quantile(arr, 0.25, axis=0)
    q3 = np.quantile(arr, 0.75, axis=0)
    iqr = q3 - q1
    iqr = np.where(iqr == 0, 1.0, iqr)
    return (arr - med) / iqr

# Robust preprocessing on both spaces
X_comp_robust = robust_scale_iqr(X_comp)
Y_sem_robust = robust_scale_iqr(Y_sem)

# Procrustes after robust scaling
X_aligned_r, Y_aligned_r, disparity_r = procrustes(X_comp_robust, Y_sem_robust)
residual_dist_r = np.linalg.norm(X_aligned_r - Y_aligned_r, axis=1)

df_procrustes_robust = pd.DataFrame({
    "name": common_names,
    "residual_distance_standard": df_procrustes.set_index("name").loc[common_names, "residual_distance"].values,
    "residual_distance_robust": residual_dist_r,
})
df_procrustes_robust["delta_robust_minus_standard"] = (
    df_procrustes_robust["residual_distance_robust"] - df_procrustes_robust["residual_distance_standard"]
)
df_procrustes_robust["abs_delta"] = df_procrustes_robust["delta_robust_minus_standard"].abs()

df_procrustes_robust = df_procrustes_robust.sort_values("residual_distance_robust", ascending=False).reset_index(drop=True)

print(f"Disparity standard : {disparity:.6f}")
print(f"Disparity robuste  : {disparity_r:.6f}")
print("\nRésumé distances robustes:")
display(df_procrustes_robust["residual_distance_robust"].describe())

print("\nTop 20 écarts (robuste):")
display(df_procrustes_robust.head(20))

print("\nTop 20 entreprises les plus sensibles au changement de normalisation:")
display(df_procrustes_robust.sort_values("abs_delta", ascending=False).head(20))

robust_path = EXPORTS_DIR / "procrustes_robust_comparison.csv"
df_procrustes_robust.to_csv(robust_path, index=False)
print(f"Export comparaison robuste → {robust_path}")

Disparity standard : 0.960928
Disparity robuste  : 0.971050

Résumé distances robustes:


count    40.000000
mean      0.137032
std       0.075096
min       0.020059
25%       0.078353
50%       0.124078
75%       0.204767
max       0.287659
Name: residual_distance_robust, dtype: float64


Top 20 écarts (robuste):


,name,residual_distance_standard,residual_distance_robust,delta_robust_minus_standard,abs_delta
0,Aqemia,0.271322,0.287659,0.016337,0.016337
1,ManoMano,0.278731,0.285299,0.006568,0.006568
2,BenevolentAI,0.256354,0.256345,-0.000009,0.000009
3,Ant Group,0.227276,0.227724,0.000447,0.000447
4,Scale AI,0.209206,0.223800,0.014594,0.014594
5,Databricks,0.213577,0.220566,0.006989,0.006989
6,UiPath,0.202791,0.217550,0.014759,0.014759
7,Prophesee,0.228591,0.211345,-0.017246,0.017246
8,Comand AI,0.212089,0.205418,-0.006670,0.006670
9,Appen,0.205524,0.205172,-0.000352,0.000352



Top 20 entreprises les plus sensibles au changement de normalisation:


,name,residual_distance_standard,residual_distance_robust,delta_robust_minus_standard,abs_delta
7,Prophesee,0.228591,0.211345,-0.017246,0.017246
0,Aqemia,0.271322,0.287659,0.016337,0.016337
21,Aleph Alpha,0.132291,0.116961,-0.015330,0.015330
6,UiPath,0.202791,0.217550,0.014759,0.014759
4,Scale AI,0.209206,0.223800,0.014594,0.014594
26,YouTube,0.111289,0.097875,-0.013414,0.013414
22,Qualcomm,0.099345,0.112541,0.013195,0.013195
11,Y combinator,0.210399,0.197764,-0.012634,0.012634
29,Cohere,0.093884,0.081360,-0.012523,0.012523
36,DeepL,0.043705,0.031832,-0.011873,0.011873


Export comparaison robuste → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\procrustes_robust_comparison.csv
